In [4]:
import pandas as pd
import numpy as np

# =========================
# 1) 데이터 불러오기
# =========================
file_path = "RXRX_stock_with_indicators.csv"
df = pd.read_csv(file_path)

# 날짜 정리
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# =========================
# 2) 볼린저 밴드폭 계산
#    Bandwidth = ((Upper - Lower) / Middle) * 100
# =========================
df["BB_WIDTH"] = ((df["BB_UPPER"] - df["BB_LOWER"]) / df["BB_MID"]) * 100

# NaN 제거 대상(초기 20일 이전 등)
valid_bw = df["BB_WIDTH"].dropna()

if valid_bw.empty:
    raise ValueError("BB_WIDTH 계산 가능한 데이터가 없습니다. BB_MID/BB_UPPER/BB_LOWER 컬럼을 확인하세요.")

# =========================
# 3) 최근 값 가져오기
# =========================
latest = df.dropna(subset=["BB_WIDTH"]).iloc[-1]

latest_date = latest["Date"].date()
latest_close = latest["Close"]
latest_bw = latest["BB_WIDTH"]
latest_rsi = latest["RSI_14"] if pd.notna(latest["RSI_14"]) else np.nan

# =========================
# 4) 역사적 비교 (분위수/평균/중앙값)
# =========================
bw_mean = valid_bw.mean()
bw_median = valid_bw.median()
bw_min = valid_bw.min()
bw_max = valid_bw.max()

# 현재 밴드폭이 과거 데이터 중 몇 퍼센트 위치인지
# 예: 3.8%면 역사적으로 매우 낮은 수준
bw_percentile = (valid_bw <= latest_bw).mean() * 100

# =========================
# 5) 최근 2주/3주/4주 추세 확인
#    미국 주식 기준 대략
#    2주=10거래일, 3주=15거래일, 4주=20거래일
# =========================
lookbacks = {
    "1주(5거래일)": 5,
    "2주(10거래일)": 10,
    "3주(15거래일)": 15,
    "4주(20거래일)": 20,
}

recent_changes = {}

for label, n in lookbacks.items():
    bw_series = df["BB_WIDTH"].dropna()
    if len(bw_series) > n:
        past_value = bw_series.iloc[-(n + 1)]
        change_abs = latest_bw - past_value
        change_pct = (change_abs / past_value) * 100 if past_value != 0 else np.nan
        recent_changes[label] = {
            "past_value": past_value,
            "current_value": latest_bw,
            "change_abs": change_abs,
            "change_pct": change_pct
        }

# =========================
# 6) 최근 며칠 데이터 테이블
# =========================
recent_table = (
    df.dropna(subset=["BB_WIDTH"])
      .tail(10)[["Date", "Close", "BB_WIDTH", "RSI_14"]]
      .copy()
)
recent_table["Date"] = recent_table["Date"].dt.date
recent_table["BB_WIDTH"] = recent_table["BB_WIDTH"].round(2)
recent_table["RSI_14"] = recent_table["RSI_14"].round(2)

# =========================
# 7) squeeze 판정 로직
#    일반 예시:
#    하위 10% 이하면 강한 squeeze
#    하위 20% 이하면 squeeze 경계
# =========================
if bw_percentile <= 10:
    squeeze_status = "강한 squeeze(변동성 압축)"
elif bw_percentile <= 20:
    squeeze_status = "squeeze 경계 구간"
else:
    squeeze_status = "특별히 압축된 구간은 아님"

# =========================
# 8) 최근 밴드폭 방향성 판정
# =========================
# 최근 5거래일 기준 기울기 느낌
last_5 = df["BB_WIDTH"].dropna().tail(5).values

if len(last_5) >= 2:
    if all(np.diff(last_5) < 0):
        bw_trend = "최근 5거래일 연속 축소"
    elif all(np.diff(last_5) > 0):
        bw_trend = "최근 5거래일 연속 확대"
    elif last_5[-1] > last_5[0]:
        bw_trend = "최근 5거래일 기준 완만한 확대"
    elif last_5[-1] < last_5[0]:
        bw_trend = "최근 5거래일 기준 완만한 축소"
    else:
        bw_trend = "최근 5거래일 큰 변화 없음"
else:
    bw_trend = "최근 추세 판정 불가"

# =========================
# 9) RSI 해석
# =========================
if pd.isna(latest_rsi):
    rsi_comment = "RSI 데이터 없음"
elif latest_rsi >= 70:
    rsi_comment = "RSI 과열권"
elif latest_rsi >= 50:
    rsi_comment = "RSI 중립 이상, 상승 모멘텀 상대적으로 양호"
elif latest_rsi >= 30:
    rsi_comment = "RSI 중립 이하, 강한 상승 모멘텀은 아직 약함"
else:
    rsi_comment = "RSI 과매도권 근처"

# =========================
# 10) 해석 출력
# =========================
print("=" * 60)
print("RXRX 볼린저 밴드폭 분석")
print("=" * 60)
print(f"기준일: {latest_date}")
print(f"종가: {latest_close:.2f}")
print(f"현재 BB Width: {latest_bw:.2f}")
print(f"과거 평균 BB Width: {bw_mean:.2f}")
print(f"과거 중앙값 BB Width: {bw_median:.2f}")
print(f"과거 최저~최고: {bw_min:.2f} ~ {bw_max:.2f}")
print(f"현재 밴드폭의 역사적 분위수: 하위 {bw_percentile:.1f}%")
print(f"상태 판정: {squeeze_status}")
print(f"최근 추세: {bw_trend}")
print(f"현재 RSI(14): {latest_rsi:.2f}" if pd.notna(latest_rsi) else "현재 RSI(14): NaN")
print(f"RSI 해석: {rsi_comment}")
print()

print("[최근 기간별 밴드폭 변화]")
for label, info in recent_changes.items():
    print(
        f"- {label}: "
        f"{info['past_value']:.2f} -> {info['current_value']:.2f} "
        f"(변화 {info['change_abs']:+.2f}, {info['change_pct']:+.2f}%)"
    )

print()
print("[최근 10거래일 데이터]")
print(recent_table.to_string(index=False))

print()
print("[자동 해석]")
if bw_percentile <= 10:
    print(
        "현재 RXRX는 볼린저 밴드폭 기준 역사적으로 매우 낮은 변동성 구간에 있습니다. "
        "즉, '압축(squeeze)' 상태로 해석할 수 있습니다."
    )
elif bw_percentile <= 20:
    print(
        "현재 RXRX는 볼린저 밴드폭이 낮은 편이며, 압축 구간에 진입했을 가능성이 있습니다."
    )
else:
    print(
        "현재 RXRX는 밴드폭 기준으로 특별히 극단적인 압축 상태는 아닙니다."
    )

if "확대" in bw_trend:
    print(
        "다만 최근 밴드폭이 다시 커지기 시작했다면, 변동성 확장 국면으로 넘어가는 초기 신호일 수 있습니다."
    )
else:
    print(
        "아직 밴드폭 자체가 본격적으로 확장되지는 않아, 방향성 돌파는 확인이 더 필요합니다."
    )

if pd.notna(latest_rsi) and latest_rsi < 50:
    print(
        "RSI가 50 아래라면, 상승 모멘텀이 강하다고 보긴 어렵고 방향성은 아직 미확정 상태로 보는 편이 안전합니다."
    )

RXRX 볼린저 밴드폭 분석
기준일: 2026-03-13
종가: 3.42
현재 BB Width: 14.58
과거 평균 BB Width: 33.41
과거 중앙값 BB Width: 31.05
과거 최저~최고: 10.05 ~ 90.09
현재 밴드폭의 역사적 분위수: 하위 4.1%
상태 판정: 강한 squeeze(변동성 압축)
최근 추세: 최근 5거래일 기준 완만한 축소
현재 RSI(14): 41.26
RSI 해석: RSI 중립 이하, 강한 상승 모멘텀은 아직 약함

[최근 기간별 밴드폭 변화]
- 1주(5거래일): 18.85 -> 14.58 (변화 -4.27, -22.65%)
- 2주(10거래일): 26.71 -> 14.58 (변화 -12.13, -45.42%)
- 3주(15거래일): 41.99 -> 14.58 (변화 -27.41, -65.28%)
- 4주(20거래일): 38.51 -> 14.58 (변화 -23.93, -62.14%)

[최근 10거래일 데이터]
      Date  Close  BB_WIDTH  RSI_14
2026-03-02   3.63     24.46   42.98
2026-03-03   3.54     21.87   40.79
2026-03-04   3.64     19.17   44.20
2026-03-05   3.54     18.39   41.62
2026-03-06   3.46     18.85   39.63
2026-03-09   3.51     16.89   41.51
2026-03-10   3.44     14.48   39.65
2026-03-11   3.45     13.09   40.06
2026-03-12   3.29     14.26   35.83
2026-03-13   3.42     14.58   41.26

[자동 해석]
현재 RXRX는 볼린저 밴드폭 기준 역사적으로 매우 낮은 변동성 구간에 있습니다. 즉, '압축(squeeze)' 상태로 해석할 수 있습니다.
아직 밴드폭 자체가 본격적으로 확장되지는 않아, 방향성

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# 1 데이터 불러오기
# ==============================

FILE_PATH = "RXRX_stock_with_indicators.csv"

df = pd.read_csv(FILE_PATH)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# ==============================
# 2 볼린저 밴드폭 계산
# ==============================

df["BB_WIDTH"] = ((df["BB_UPPER"] - df["BB_LOWER"]) / df["BB_MID"]) * 100

# ==============================
# 3 추가 지표 계산
# ==============================

df["VOL_MA20"] = df["Volume"].rolling(20).mean()
df["VOL_RATIO_20"] = df["Volume"] / df["VOL_MA20"]

df["BW_DELTA"] = df["BB_WIDTH"].diff()

# 최근 3일 밴드폭 확장 여부
df["BW_EXPANSION_3D"] = (
    (df["BB_WIDTH"] > df["BB_WIDTH"].shift(1)) &
    (df["BB_WIDTH"].shift(1) > df["BB_WIDTH"].shift(2))
)

# ==============================
# 4 squeeze 판단
# ==============================

lookback = 120

def compute_percentile(series):
    current = series.iloc[-1]
    return (series <= current).mean() * 100

percentiles = []

for i in range(len(df)):
    if i < lookback:
        percentiles.append(np.nan)
    else:
        window = df["BB_WIDTH"].iloc[i-lookback:i]
        pct = (window <= df["BB_WIDTH"].iloc[i]).mean() * 100
        percentiles.append(pct)

df["BW_PERCENTILE_120"] = percentiles

df["SQUEEZE"] = df["BW_PERCENTILE_120"] <= 10

# ==============================
# 5 돌파 신호
# ==============================

df["UPPER_BREAK"] = df["Close"] > df["BB_UPPER"]
df["LOWER_BREAK"] = df["Close"] < df["BB_LOWER"]

# ==============================
# 6 트레이딩 시그널
# ==============================

signals = []

for i,row in df.iterrows():

    signal = "Neutral"

    if row["SQUEEZE"]:
        signal = "Squeeze watch"

    if row["UPPER_BREAK"] and row["VOL_RATIO_20"] > 1.5 and row["RSI_14"] > 50:
        signal = "Bullish breakout"

    if row["LOWER_BREAK"] and row["VOL_RATIO_20"] > 1.5 and row["RSI_14"] < 45:
        signal = "Bearish breakdown"

    if row["BW_EXPANSION_3D"] and signal == "Neutral":
        signal = "Expansion without full confirmation"

    signals.append(signal)

df["SIGNAL"] = signals

# ==============================
# 7 최근 변화 분석
# ==============================

latest = df.dropna(subset=["BB_WIDTH"]).iloc[-1]

lookbacks = {
    "1W":5,
    "2W":10,
    "3W":15,
    "4W":20
}

changes = []

for label,n in lookbacks.items():

    if len(df) > n:

        past = df["BB_WIDTH"].iloc[-n-1]
        now = latest["BB_WIDTH"]

        diff = now - past
        pct = diff/past*100

        changes.append({
            "Period":label,
            "Past_BW":past,
            "Current_BW":now,
            "Change":diff,
            "Change_%":pct
        })

changes_df = pd.DataFrame(changes)

# ==============================
# 8 Summary 생성
# ==============================

summary = pd.DataFrame({
    "Metric":[
        "Latest Date",
        "Close",
        "BB Width",
        "Bandwidth Percentile(120d)",
        "RSI",
        "Volume Ratio",
        "Current Signal"
    ],
    "Value":[
        latest["Date"],
        latest["Close"],
        latest["BB_WIDTH"],
        latest["BW_PERCENTILE_120"],
        latest["RSI_14"],
        latest["VOL_RATIO_20"],
        latest["SIGNAL"]
    ]
})

# ==============================
# 9 그래프 생성
# ==============================

# 가격 + 볼린저 밴드
plt.figure(figsize=(12,6))
plt.plot(df["Date"],df["Close"],label="Close")
plt.plot(df["Date"],df["BB_UPPER"],label="Upper")
plt.plot(df["Date"],df["BB_MID"],label="Middle")
plt.plot(df["Date"],df["BB_LOWER"],label="Lower")
plt.title("RXRX Price and Bollinger Bands")
plt.legend()
plt.savefig("rxrx_price_bbands.png")
plt.close()

# 밴드폭
plt.figure(figsize=(12,6))
plt.plot(df["Date"],df["BB_WIDTH"])
plt.title("Bollinger Bandwidth")
plt.savefig("rxrx_bandwidth.png")
plt.close()

# RSI
plt.figure(figsize=(12,6))
plt.plot(df["Date"],df["RSI_14"])
plt.axhline(70)
plt.axhline(30)
plt.title("RSI 14")
plt.savefig("rxrx_rsi.png")
plt.close()

# ==============================
# 10 CSV 저장
# ==============================

df.to_csv("RXRX_analysis_output.csv",index=False)

# ==============================
# 11 엑셀 생성
# ==============================

excel_path = "RXRX_bollinger_analysis.xlsx"

with pd.ExcelWriter(excel_path,engine="openpyxl") as writer:

    summary.to_excel(writer,sheet_name="Summary",index=False)
    changes_df.to_excel(writer,sheet_name="Recent_Changes",index=False)
    df.tail(60).to_excel(writer,sheet_name="Signal_Log",index=False)
    df.to_excel(writer,sheet_name="Analyzed_Data",index=False)

print("분석 완료")
print("엑셀 파일 생성:",excel_path)
print("그래프 파일 생성 완료")